# 🤖 Nova — Personal AI Assistant
### End-to-End Python Walkthrough (Local Jupyter)

This notebook walks through every module of **Nova** step by step.
You can run and test each skill independently before running the full assistant.

**Architecture overview:**
```
         ┌─────────────────────────────┐
         │         main.py             │
         │  Wake loop → Route → Speak  │
         └────────────┬────────────────┘
              ┌───────┴────────┐
           stt.py          tts.py
        (hear you)      (speak back)
              └───────┬────────┘
                  router.py
            (intent detection)
          ┌──────────┴──────────┐
       skills.py           brain.py
    (apps/music/           (Gemini /
     weather/stocks)      OpenRouter)
```

## 📦 Step 1 — Install Dependencies

In [ ]:
# Run this cell once to install everything
import subprocess, sys

packages = [
    'SpeechRecognition',
    'pyttsx3',
    'gTTS',
    'pygame',
    'soundfile',
    'numpy',
    'requests',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ Core packages installed!')
print()
print('📝 For PyAudio (microphone support) on Windows:')
print('   pip install pipwin')
print('   pipwin install pyaudio')
print()
print('📝 For Whisper (optional, offline STT):')
print('   pip install openai-whisper')

## ⚙️ Step 2 — Configuration

In [ ]:
# ── Paste your keys here for notebook testing ──
GEMINI_API_KEY      = 'YOUR_GEMINI_API_KEY'       # https://aistudio.google.com
OPENROUTER_API_KEY  = 'YOUR_OPENROUTER_API_KEY'   # https://openrouter.ai
OPENWEATHER_API_KEY = 'YOUR_OPENWEATHER_API_KEY'  # https://openweathermap.org
DEFAULT_CITY        = 'Lagos'
WAKE_WORD           = 'hey nova'

print('Configuration loaded.')

## 🔊 Step 3 — Text-to-Speech (TTS)

In [ ]:
# Test gTTS (needs internet)
import os, tempfile
from gtts import gTTS
import pygame

def speak_gtts(text):
    tts = gTTS(text=text, lang='en', slow=False)
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        path = f.name
    tts.save(path)
    pygame.mixer.init()
    pygame.mixer.music.load(path)
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        pygame.time.Clock().tick(10)
    pygame.mixer.music.unload()
    os.remove(path)
    print(f'Nova said: "{text}"')

speak_gtts('Hello! I am Nova, your personal assistant.')

In [ ]:
# Test pyttsx3 fallback (offline)
import pyttsx3

def speak_pyttsx3(text):
    engine = pyttsx3.init()
    engine.setProperty('rate', 175)
    engine.say(text)
    engine.runAndWait()
    print(f'Nova said (pyttsx3): "{text}"')

speak_pyttsx3('Offline voice test successful.')

## 🎤 Step 4 — Speech-to-Text (STT)

In [ ]:
# Test Google Speech Recognition (needs internet + microphone)
import speech_recognition as sr

recognizer = sr.Recognizer()
mic        = sr.Microphone()

print('Calibrating microphone for ambient noise (1 sec)...')
with mic as source:
    recognizer.adjust_for_ambient_noise(source, duration=1)

print('🎙️ Speak now! (5 second window)')
with mic as source:
    audio = recognizer.listen(source, timeout=5, phrase_time_limit=8)

try:
    text = recognizer.recognize_google(audio)
    print(f'✅ You said: "{text}"')
except sr.UnknownValueError:
    print('Could not understand audio')
except sr.RequestError as e:
    print(f'API error: {e}')

## 🧠 Step 5 — AI Brain (Gemini)

In [ ]:
import json, urllib.request

def ask_gemini(question, api_key=GEMINI_API_KEY):
    if api_key == 'YOUR_GEMINI_API_KEY':
        return 'Gemini API key not set. Add your key above.'
    payload = json.dumps({
        'contents': [{'role': 'user', 'parts': [{'text': question}]}],
        'generationConfig': {'maxOutputTokens': 200}
    }).encode()
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}'
    req = urllib.request.Request(url, data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read())
    return data['candidates'][0]['content']['parts'][0]['text'].strip()

answer = ask_gemini('What is the capital of Nigeria?')
print(f'Gemini says: {answer}')

## 💻 Step 6 — Open App Skill

In [ ]:
import subprocess, shutil

KNOWN_APPS = {
    'notepad': 'notepad.exe',
    'calculator': 'calc.exe',
    'chrome': 'chrome.exe',
    'paint': 'mspaint.exe',
    'explorer': 'explorer.exe',
}

def open_app(app_name):
    exe = KNOWN_APPS.get(app_name.lower())
    if exe:
        subprocess.Popen(exe, shell=True)
        return f'Opening {app_name}...'
    return f"I don't know how to open '{app_name}'"

# Test — opens Notepad
result = open_app('notepad')
print(result)

## 🎵 Step 7 — Music Skills

In [ ]:
import webbrowser, urllib.parse

def play_youtube(query):
    url = f'https://www.youtube.com/results?search_query={urllib.parse.quote(query)}'
    webbrowser.open(url)
    return f'Searching YouTube for: {query}'

# Test
print(play_youtube('Afrobeats 2024 mix'))

## 🌤️ Step 8 — Weather Skill

In [ ]:
import json, urllib.request, urllib.parse

def get_weather(city, api_key=OPENWEATHER_API_KEY):
    if api_key == 'YOUR_OPENWEATHER_API_KEY':
        return 'OpenWeatherMap API key not set.'
    url = (f'https://api.openweathermap.org/data/2.5/weather'
           f'?q={urllib.parse.quote(city)}&appid={api_key}&units=metric')
    with urllib.request.urlopen(url, timeout=10) as resp:
        data = json.loads(resp.read())
    desc  = data['weather'][0]['description'].capitalize()
    temp  = data['main']['temp']
    feels = data['main']['feels_like']
    return f'{city}: {desc}, {temp:.1f}°C (feels like {feels:.1f}°C)'

print(get_weather(DEFAULT_CITY))

## 📈 Step 9 — Stock Lookup

In [ ]:
import webbrowser

def lookup_stock(ticker):
    url = f'https://finance.yahoo.com/quote/{ticker.upper()}'
    webbrowser.open(url)
    return f'Opening Yahoo Finance for {ticker.upper()}'

print(lookup_stock('AAPL'))

## 🔍 Step 10 — Web Search

In [ ]:
import webbrowser, urllib.parse

def web_search(query):
    url = f'https://www.google.com/search?q={urllib.parse.quote(query)}'
    webbrowser.open(url)
    return f'Searching Google for: {query}'

print(web_search('Python asyncio tutorial'))

## 🚀 Step 11 — Run Nova (Full Session)

This cell runs Nova interactively in the notebook.
It uses **text input** instead of voice (mic doesn't work well in Jupyter).
For the full voice experience, run `python main.py` from your terminal.

In [ ]:
import sys, os
sys.path.insert(0, 'nova')   # adjust path to where you saved the project

# Simple text-mode Nova for notebook testing
import importlib

print('Nova Text Mode (type quit to exit)\n')

def nova_text_session():
    from nova import router, tts
    while True:
        command = input('You: ').strip()
        if not command:
            continue
        response = router.route(command)
        if response == '__EXIT__':
            print('Nova: Goodbye!')
            break
        print(f'Nova: {response}\n')

nova_text_session()